**This notebook we use as a common functions in entire project**

In [1]:

# Standard Python libraries
import os
import re
import zipfile
import smtplib
import time
import traceback
import json
import yaml
import posixpath
import uuid

from io import BytesIO
from datetime import datetime, timezone
from functools import reduce
from html import escape
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# Snowflake Snowpark
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.functions import col, lit

session = get_active_session()

print("Common function imports loaded successfully!")


In [ ]:
%run ../utility/variable_assignments.ipynb

In [1]:

# # Get the Snowflake user executing the notebook
# run_by_user = session.sql(
#     "SELECT CURRENT_USER()"
# ).collect()[0][0]

# print("Pipeline executed by:", run_by_user)


In [ ]:
%%sql -r dataframe_1
-- LIST @CUSTOMER_BILLING_DB.PUBLIC.AZURE_SNOW_STAGE/inbound/customer_billing/;

In [ ]:
# Read inbound folder from configuration
# Convert configured stage reference into Snowflake stage path
input_stage_path = input_folder.replace("@AZURE_SNOW_STAGE", f"@{stage_name}")

# Discover ZIP files dynamically
zip_files = session.sql(f"LIST {input_stage_path}").collect()
zip_files = [
    row["name"]
    for row in zip_files
    if row["name"].lower().endswith(".zip")
]
print("ZIP files discovered:", len(zip_files))

for file_name in zip_files:
    print(file_name)

In [3]:
def unzip_files(zip_path, unzip_path, expected_files):
    validation_messages = []
    # 1. Normalize configured stage paths
    if zip_path.startswith("@AZURE_SNOW_STAGE"):
        zip_path = zip_path.replace("@AZURE_SNOW_STAGE", "@CUSTOMER_BILLING_DB.PUBLIC.AZURE_SNOW_STAGE", 1)

    if unzip_path.startswith("@AZURE_SNOW_STAGE"):
        unzip_path = unzip_path.replace("@AZURE_SNOW_STAGE", "@CUSTOMER_BILLING_DB.PUBLIC.AZURE_SNOW_STAGE", 1)

    zip_path = zip_path.rstrip("/")
    unzip_path = unzip_path.rstrip("/") + "/"

    if not zip_path.startswith("@") or not unzip_path.startswith("@"):
        raise ValueError("zip_path and unzip_path must be Snowflake stage paths.")

    source_directory = zip_path.rsplit("/", 1)[0] + "/"
    zip_filename = zip_path.rsplit("/", 1)[-1]

    if not zip_filename.lower().endswith(".zip"):
        raise ValueError(f"Not a ZIP file: {zip_filename}")

    # 2. Validate ZIP existence in external stage
    source_listing = session.sql(f"LIST {source_directory}").collect()
    source_exists = any(str(row["name"]).rstrip("/").endswith("/" + zip_filename)
        for row in source_listing
    )

    if not source_exists:
        raise FileNotFoundError(f"ZIP file not found in stage: {zip_path}")
    print(f"ZIP file found: {zip_filename}")
    validation_messages.append(f"ZIP file found: {zip_filename}")

    # 3. Create a temporary internal stage
    temp_stage_name = ("CUSTOMER_BILLING_DB.PUBLIC.CB_PIPELINE_TEMP_STAGE")
    session.sql(f"CREATE TEMP STAGE IF NOT EXISTS {temp_stage_name}").collect()
    unique_id = uuid.uuid4().hex
    source_temp_path = (f"@{temp_stage_name}/source/{unique_id}/")
    extracted_temp_path = (f"@{temp_stage_name}/extracted/{unique_id}/")

    # 4. Copy ZIP from external stage to internal stage
    escaped_filename = zip_filename.replace("'", "''")
    session.sql(f"""COPY FILES INTO {source_temp_path} FROM {source_directory} FILES = ('{escaped_filename}')""").collect()
    internal_zip_path = (f"{source_temp_path}{zip_filename}")

    # 5. Read ZIP from internal stage
    with session.file.get_stream(internal_zip_path) as zip_stream:
        with zipfile.ZipFile(zip_stream, mode="r") as zip_reference:
            
            # Validation 1: ZIP contains files
            zip_entries = [
                item for item in zip_reference.infolist()
                if not item.is_dir()
            ]
            if len(zip_entries) == 0:
                raise ValueError(f"ZIP file is empty: {zip_filename}")
            print(f"ZIP contains {len(zip_entries)} files.")
            validation_messages.append(f"ZIP contains {len(zip_entries)} files.")

            # Validation 2: Identify empty files
            empty_files = [
                item.filename
                for item in zip_entries
                if item.file_size == 0
            ]

            if empty_files:
                print("Empty files found:", empty_files)
                validation_messages.append(f"Empty files found: {empty_files}")
            else:
                print("No empty files found.")
                validation_messages.append("No empty files found.")

            # Validation 3: Check ZIP integrity
            corrupted_file = zip_reference.testzip()
            if corrupted_file:
                raise ValueError(f"Corrupted file inside ZIP: {corrupted_file}")
            print("ZIP integrity validation passed.")
            validation_messages.append("ZIP integrity validation passed.")

            # Validation 4: Expected file count
            total_files = len(zip_entries)
            print("Actual files:", total_files)
            print("Expected files:", expected_files)
            if total_files != int(expected_files):
                missing_count = (int(expected_files) - total_files)
                count_message = (
                    f"File count mismatch. "
                    f"Expected: {expected_files}, "
                    f"Actual: {total_files}, "
                    f"Difference: {missing_count}"
                )
                print(count_message)
                validation_messages.append(count_message)
            else:
                print("Expected file count validation passed.")
                validation_messages.append("Expected file count validation passed.")


            # 6. Generate timestamped extraction folder
            folder_name = datetime.now().strftime("%Y%m%d_%H%M%S")
            actual_unzip_path = (f"{unzip_path}{folder_name}/")
            print("Extraction destination:", actual_unzip_path)

            # 7. Extract and upload files
            extracted_files = []
            for file_info in zip_entries:
                relative_path = file_info.filename.replace("\\", "/")
                normalized_path = posixpath.normpath(relative_path)

                # Prevent ZIP path traversal
                if (
                    normalized_path in ("", ".")
                    or normalized_path.startswith("/")
                    or normalized_path == ".."
                    or normalized_path.startswith("../")
                    or ":" in normalized_path.split("/")[0]
                ):
                    raise ValueError(f"Unsafe ZIP entry: {file_info.filename}")

                # Preserve directory structure
                destination_file = (f"{actual_unzip_path}{normalized_path}")

                # Read file contents from ZIP
                with zip_reference.open(file_info, "r") as extracted_stream:
                    file_bytes = extracted_stream.read()

                # Upload to internal temporary stage
                internal_file_path = (f"{extracted_temp_path}{normalized_path}")
                session.file.put_stream(
                    BytesIO(file_bytes),
                    internal_file_path,
                    auto_compress=False,
                    overwrite=True
                )
                extracted_files.append(normalized_path)
                print(f"Prepared for extraction: {normalized_path}")


            # 8. Copy extracted files to Azure external stage
            session.sql(f"""COPY FILES INTO {actual_unzip_path} FROM {extracted_temp_path}""").collect()
            print("All extracted files uploaded successfully.")
            validation_messages.append(f"Extraction completed: {actual_unzip_path}")

    # 9. Return validation summary
    validation_message = "\n".join(validation_messages)
    print("\n========== ZIP PROCESSING SUMMARY ==========")
    print(validation_message)
    print("============================================")
    return validation_message

In [4]:
# Snowflake ETL Logging Function
def process_log(run_status,file_name=None,table_name=None,layer=None,total_inserted_count=None,total_updated_count=None,error_message=None,email_status=None,business_date=None,run_end_time=None):
    global run_id, run_start_time, pipeline_name, task_name

    # Get current Snowflake notebook session
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()

    # Capture execution end time
    run_end_time = run_end_time or datetime.now()

    # Set business date
    business_date = business_date or run_end_time.date()

    # Validate required execution variables
    if not run_id:
        raise ValueError("run_id is not defined")

    if not run_start_time:
        raise ValueError("run_start_time is not defined")

    # Insert or update ETL log record
    merge_sql = """
        MERGE INTO CUSTOMER_BILLING_DB.CONTROL.ETL_RUN_LOG AS T
        USING (
            SELECT? AS RUN_ID,? AS PIPELINE_NAME,? AS TASK_NAME,? AS FILE_NAME,? AS TABLE_NAME,? AS LAYER,? AS BUSINESS_DATE,? AS RUN_START_TIME,? AS RUN_END_TIME,? AS RUN_STATUS,? AS TOTAL_INSERTED_COUNT,? AS TOTAL_UPDATED_COUNT,? AS ERROR_MESSAGE,? AS EMAIL_STATUS
        ) AS S
        ON T.RUN_ID = S.RUN_ID

        WHEN MATCHED THEN UPDATE SET
            T.RUN_STATUS = S.RUN_STATUS,
            T.RUN_END_TIME = S.RUN_END_TIME,
            T.TOTAL_INSERTED_COUNT =
                COALESCE(S.TOTAL_INSERTED_COUNT, T.TOTAL_INSERTED_COUNT),
            T.TOTAL_UPDATED_COUNT =
                COALESCE(S.TOTAL_UPDATED_COUNT, T.TOTAL_UPDATED_COUNT),
            T.ERROR_MESSAGE = S.ERROR_MESSAGE,
            T.EMAIL_STATUS =
                COALESCE(S.EMAIL_STATUS, T.EMAIL_STATUS)

        WHEN NOT MATCHED THEN INSERT (RUN_ID,PIPELINE_NAME,TASK_NAME,FILE_NAME,TABLE_NAME,LAYER,BUSINESS_DATE,RUN_START_TIME,RUN_END_TIME,RUN_STATUS,TOTAL_INSERTED_COUNT,TOTAL_UPDATED_COUNT,ERROR_MESSAGE,EMAIL_STATUS
        )
        VALUES (S.RUN_ID,S.PIPELINE_NAME,S.TASK_NAME,S.FILE_NAME,S.TABLE_NAME,S.LAYER,S.BUSINESS_DATE,S.RUN_START_TIME,S.RUN_END_TIME,S.RUN_STATUS,S.TOTAL_INSERTED_COUNT,S.TOTAL_UPDATED_COUNT,S.ERROR_MESSAGE,S.EMAIL_STATUS
        )
    """

    params = [run_id,pipeline_name,task_name,file_name,table_name,layer,business_date,run_start_time,run_end_time,run_status,total_inserted_count,total_updated_count,error_message,email_status]
    session.sql(merge_sql, params=params).collect()
    print(f"ETL log updated successfully: "f"{run_status} | {file_name}")

In [6]:
def open_file_read_data(file_path):
    session = get_active_session()

    # Convert Azure LIST URL to a Snowflake stage path
    if file_path.startswith("azure://") or file_path.startswith("https://"):
        if "/snow/" not in file_path:
            raise ValueError(f"Cannot convert Azure URL to stage path: {file_path}")

        relative_path = file_path.split("/snow/", 1)[1]
        stage_file_path = (
            "@CUSTOMER_BILLING_DB.PUBLIC.AZURE_SNOW_STAGE/" + relative_path
        )
    else:
        stage_file_path = file_path

    # Create a temporary format that reads each text line as one field
    session.sql("""
        CREATE OR REPLACE TEMP FILE FORMAT CB_TEXT_LINE_FORMAT
        TYPE = CSV
        FIELD_DELIMITER = NONE
    """).collect()

    # Read the staged file
    query = f"""
        SELECT $1
        FROM {stage_file_path}
        (FILE_FORMAT => 'CB_TEXT_LINE_FORMAT')
    """

    result = session.sql(query).collect()

    file_content = "\n".join(
        str(row[0]) for row in result if row[0] is not None
    )

    if not file_content:
        raise FileNotFoundError(f"File is empty or not found: {file_path}")

    return file_content

In [7]:
def read_recursive_file_data(file_path):
    session = get_active_session()
    all_files = session.sql(f"LIST {file_path}").collect()
    for actual_file_path in all_files:
        file_name = actual_file_path["name"].rsplit("/", 1)[-1]
        if file_name.startswith(file_name_prefix):
            full_path = actual_file_path["name"]
            if "/snow/" not in full_path:
                raise ValueError(f"Unexpected Azure file path: {full_path}")
            relative_path = full_path.split("/snow/", 1)[1]
            final_file_name = file_name
            final_file_path = (
                "@CUSTOMER_BILLING_DB.PUBLIC.AZURE_SNOW_STAGE/"
                + relative_path
            )
            return final_file_name, final_file_path
    return None, None

In [8]:
#Match file name patttern
def match_file_name_pattern(file_name_patern, file_name, date_time_format=None):
    is_matched = re.match(file_name_pattern, file_name)
    if date_time_format:
        date_time_format = date_time_format
    else:
        date_time_format = "%Y%m%d%H%M%S"
    if is_matched:
        try:
            if datetime.strptime(f"{is_matched.group(1)}{is_matched.group(2)}", date_time_format):
                file_name_validation_message = "Date is valid"
        except ValueError:
            file_name_validation_message = "Date is invalid"
    else:
        file_name_validation_message =  "File format is wrong"
    return file_name_validation_message

In [ ]:
#Reusable functions from silver layer
#Generate hash key
def generate_hash_key(df, hash_columns, hash_name="hash_key"):
    payload = F.to_json(
        F.struct(*[F.col(c).cast("string").alias(c) for c in hash_columns]),
        options={"ignoreNullFields": "false"}
    )
    return df.withColumn(hash_name, F.sha2(payload, 256))

#Validate duplicates based on business key
def validate_business_keys(df, business_keys, table_name):
    for key in business_keys:
        if df.filter(F.col(key).isNull()).limit(1).count():
            raise ValueError(f"Null business key found in {table_name}: {key}")

    if df.groupBy(*business_keys).count().filter(F.col("count") > 1).limit(1).count():
        raise ValueError(f"Duplicate business keys found in {table_name}")

#Get count after succesful
def get_delta_merge_counts(table_name):
    history = DeltaTable.forName(spark, table_name).history(1).first()

    if history is None or history["operation"] != "MERGE":
        raise ValueError(f"No MERGE operation found in Delta history for {table_name}")

    metrics = history["operationMetrics"] or {}
    inserted_count = int(metrics.get("numTargetRowsInserted", "0"))
    updated_count = int(metrics.get("numTargetRowsUpdated", "0"))

    return inserted_count, updated_count

In [ ]:
# Function to log duplicate records
def duplicate_log(duplicate_df, run_id, source_table, business_keys, data_columns):
    duplicate_log_table = "control.duplicate_data_log"

    # Check whether duplicate records exist
    if duplicate_df.limit(1).count() == 0:
        print(f"No duplicate records found in {source_table}")
        return 0

    # Prepare duplicate log data
    duplicate_log_df = duplicate_df.select(
        F.lit(run_id).alias("run_id"),
        F.lit(source_table).alias("source_table"),
        F.lit(",".join(business_keys)).alias("business_key"),
        F.to_json(
            F.struct(*[F.col(c) for c in business_keys])
        ).alias("business_key_value"),
        F.col("duplicate_count").cast("int"),
        F.to_json(
            F.struct(*[F.col(c) for c in data_columns])
        ).alias("record_data"),
        F.current_timestamp().alias("logged_at")
    )

    # Insert duplicate records into control table
    duplicate_log_df.write.format("delta").mode("append").saveAsTable(duplicate_log_table)
    duplicate_count = duplicate_df.count()
    print(f"Duplicate records logged successfully: "f"{duplicate_count} | Table: {duplicate_log_table}")
    return duplicate_count

In [ ]:
#Send email notification function
sender_email = "amit.aigk@gmail.com"
to_email = "amit.aigk@gmail.com"
gmail_app_password = "djvj girr ycun xrqp"
host = "smtp.gmail.com"
port = 587
def send_mail_fn(f_subject, b_message):
    safe_subject = escape(str(f_subject))
    safe_message = (escape(str(b_message)).replace("\r\n", "\n").replace("\n", "<br>"))
    html_template = f"""<!DOCTYPE html><html><head><meta charset="UTF-8"></head><body style=" margin:0; padding:20px; background-color:#f4f6f9; font-family:Arial,Helvetica,sans-serif; "><table width="100%" cellpadding="0" cellspacing="0" style="max-width:650px;margin:auto; background-color:#ffffff;border-collapse:collapse;"><tr><td style=" background-color:#17365D; padding:22px; text-align:center; color:#ffffff; "><h2 style="margin:0;"> Pipeline Execution Notification </h2><p style="margin:8px 0 0;font-size:13px;"> Customer Billing Data Platform </p></td></tr><tr><td style=" padding:25px; color:#333333; font-size:14px; line-height:1.7; "><h3 style="margin-top:0;color:#17365D;"> {safe_subject} </h3><div style=" background-color:#f4f6f9; border-left:4px solid #17365D; padding:15px; border-radius:4px; overflow-wrap:anywhere; "> {safe_message} </div></td></tr><tr><td style=" background-color:#f1f3f5; padding:15px; text-align:center; color:#777777; font-size:12px; "> This is an automated email notification.<br> Data Engineering Team </td></tr></table></body></html>"""
    server = None
    try:
        mail = MIMEMultipart("alternative")
        mail["Subject"] = str(f_subject)
        mail["From"] = sender_email
        mail["To"] = to_email
        mail.attach(MIMEText(html_template, "html", "utf-8"))
        server = smtplib.SMTP(host, port, timeout=30)
        server.ehlo()
        server.starttls()
        server.ehlo()
        server.login(sender_email, gmail_app_password)
        server.send_message(mail)
        print("Email sent successfully!")
    except Exception as e:
        print("Email sending failed:", str(e))
        raise
    finally:
        if server is not None:
            try:
                server.quit()
            except Exception:
                pass
def send_pipeline_notification_fn(pipeline_run_id, pipeline_name, pipeline_start_time=None, log_table="control.etl_run_log"):
    if not pipeline_run_id or not str(pipeline_run_id).strip():
        raise ValueError("pipeline_run_id is required.")
    if not pipeline_name or not str(pipeline_name).strip():
        raise ValueError("pipeline_name is required.")
    required_columns = [
        "run_id", "pipeline_name", "task_name", "file_name", "table_name",
        "layer", "business_date", "run_start_time", "run_end_time",
        "run_status", "total_inserted_count", "total_updated_count",
        "error_message", "email_status"
    ]
    log_df = spark.table(log_table)
    missing_columns = [c for c in required_columns if c not in log_df.columns]
    if missing_columns:
        raise ValueError(
            f"Table {log_table} is missing columns: {', '.join(missing_columns)}"
        )
    run_records = (
        log_df
        .filter(log_df["run_id"] == str(pipeline_run_id))
        .orderBy("run_start_time", "layer", "task_name")
        .collect()
    )
    if not run_records:
        raise RuntimeError(
            f"No records found in {log_table} for run_id={pipeline_run_id}. "
            "Verify that all layer notebooks log the same pipeline RunId."
        )
    def _value(value, default="N/A"):
        if value is None:
            return default
        value = str(value).strip()
        return value if value else default
    def _category(status):
        value = _value(status, "").lower()
        if any(word in value for word in ("fail", "error", "exception")):
            return "failed"
        if any(word in value for word in (
            "success", "succeeded", "complete", "completed"
        )):
            return "success"
        if any(word in value for word in ("running", "in progress", "started")):
            return "running"
        return "other"
    def _format_time(value):
        return value.strftime("%Y-%m-%d %H:%M:%S") if value else "N/A"
    total_tasks = len(run_records)
    counts = {"success": 0, "failed": 0, "running": 0, "other": 0}
    total_inserted = 0
    total_updated = 0
    layer_summary = {}
    activity_details = []
    failed_details = []
    start_times = []
    end_times = []
    for row in run_records:
        category = _category(row["run_status"])
        layer = _value(row["layer"], "Unknown")
        counts[category] += 1
        inserted = int(row["total_inserted_count"] or 0)
        updated = int(row["total_updated_count"] or 0)
        total_inserted += inserted
        total_updated += updated
        if layer not in layer_summary:
            layer_summary[layer] = {
                "total": 0,
                "success": 0,
                "failed": 0,
                "running": 0,
                "other": 0,
                "inserted": 0,
                "updated": 0
            }
        layer_summary[layer]["total"] += 1
        layer_summary[layer][category] += 1
        layer_summary[layer]["inserted"] += inserted
        layer_summary[layer]["updated"] += updated
        start_value = row["run_start_time"]
        end_value = row["run_end_time"]
        if start_value is not None:
            start_times.append(start_value)
        if end_value is not None:
            end_times.append(end_value)
        detail = {
            "layer": layer,
            "task": _value(row["task_name"]),
            "file": _value(row["file_name"]),
            "table": _value(row["table_name"]),
            "status": _value(row["run_status"]),
            "start": _format_time(start_value),
            "end": _format_time(end_value),
            "inserted": inserted,
            "updated": updated,
            "error": _value(row["error_message"], "")
        }
        activity_details.append(detail)
        if category == "failed":
            failed_details.append(detail)
    overall_start = min(start_times) if start_times else None
    overall_end = max(end_times) if end_times else None
    if pipeline_start_time:
        try:
            parsed_start = datetime.fromisoformat(
                str(pipeline_start_time).replace("Z", "+00:00")
            )
            if parsed_start.tzinfo is not None and overall_end is not None:
                parsed_start = parsed_start.replace(tzinfo=None)
            overall_start = parsed_start
        except (TypeError, ValueError):
            print(
                "Warning: pipeline_start_time could not be parsed. "
                "Using the earliest run_start_time in the log table."
            )
    duration_text = "Not available"
    if overall_start is not None and overall_end is not None:
        if overall_start.tzinfo is not None and overall_end.tzinfo is None:
            overall_start = overall_start.replace(tzinfo=None)
        elif overall_start.tzinfo is None and overall_end.tzinfo is not None:
            overall_end = overall_end.replace(tzinfo=None)
        seconds = max(0, int((overall_end - overall_start).total_seconds()))
        hours, remainder = divmod(seconds, 3600)
        minutes, seconds = divmod(remainder, 60)
        duration_text = f"{hours:02d}:{minutes:02d}:{seconds:02d}"
    if counts["failed"] > 0:
        overall_status = "FAILED"
    elif counts["running"] > 0 or counts["other"] > 0:
        overall_status = "COMPLETED WITH NON-SUCCESS TASKS"
    else:
        overall_status = "SUCCESS"
    subject = f"{pipeline_name} - {overall_status} - Pipeline Execution Summary"
    lines = [
        "PIPELINE EXECUTION SUMMARY",
        "=" * 34,
        f"Pipeline Name : {pipeline_name}",
        f"Pipeline Run ID: {pipeline_run_id}",
        f"Overall Status: {overall_status}",
        f"Pipeline Start: {_format_time(overall_start)}",
        f"Pipeline End  : {_format_time(overall_end)}",
        f"Total Duration: {duration_text}",
        "",
        "OVERALL COUNTS",
        "-" * 34,
        f"Total activity records : {total_tasks}",
        f"Successful             : {counts['success']}",
        f"Failed                 : {counts['failed']}",
        f"Running                : {counts['running']}",
        f"Other / unknown        : {counts['other']}",
        f"Total inserted records : {total_inserted:,}",
        f"Total updated records  : {total_updated:,}",
        "",
        "LAYER SUMMARY",
        "-" * 34
    ]
    for layer in sorted(layer_summary):
        item = layer_summary[layer]
        lines.append(
            f"{layer}: total={item['total']}, success={item['success']}, "
            f"failed={item['failed']}, running={item['running']}, "
            f"other={item['other']}, inserted={item['inserted']:,}, "
            f"updated={item['updated']:,}"
        )
    lines.extend(["", "ACTIVITY DETAILS", "-" * 34])
    for item in activity_details:
        lines.append(
            f"Layer: {item['layer']} | Task: {item['task']} | "
            f"File: {item['file']} | Table: {item['table']}"
        )
        lines.append(
            f"Status: {item['status']} | Start: {item['start']} | "
            f"End: {item['end']} | Inserted: {item['inserted']:,} | "
            f"Updated: {item['updated']:,}"
        )
        if item["error"]:
            lines.append(f"Error: {item['error']}")
        lines.append("")
    if failed_details:
        lines.extend(["FAILED ACTIVITY DETAILS", "-" * 34])
        for item in failed_details:
            lines.append(
                f"{item['layer']} | {item['task']} | "
                f"File: {item['file']} | Table: {item['table']}"
            )
            lines.append(f"Error: {item['error'] or 'No error_message recorded'}")
            lines.append("")
    email_body = "\n".join(lines)
    send_mail_fn(subject, email_body)
    print(
        f"Pipeline notification sent. Run ID={pipeline_run_id}, "
        f"status={overall_status}, records={total_tasks}."
    )
    return {
        "run_id": str(pipeline_run_id),
        "pipeline_name": str(pipeline_name),
        "overall_status": overall_status,
        "total_tasks": total_tasks,
        "success_count": counts["success"],
        "failed_count": counts["failed"],
        "running_count": counts["running"],
        "other_count": counts["other"],
        "total_inserted_count": total_inserted,
        "total_updated_count": total_updated,
        "duration": duration_text,
        "layer_summary": layer_summary
    }